# MiroFish fully inside Google Colab

This notebook has **no Zep account, Zep API key, Gemini key, or external inference service**. It runs the local-first MiroFish fork with Graphiti + Neo4j inside the Colab VM, `LiquidAI/LFM2.5-350M` on the Colab GPU (vLLM, OpenAI-compatible) for chat/completions, and `BAAI/bge-m3` locally for embeddings.

It downloads public code/model weights from GitHub, npm, PyPI and Hugging Face at setup time, and Cloudflare Quick Tunnels expose only the UI and the Flask API to your browser — and only at the very end, after the pipeline is proven locally. Seed data, graph data, embeddings, and LLM prompts stay in the Colab runtime.

## Important design change

The upstream `666ghj/MiroFish` requires Zep Cloud. This notebook uses `tt-a1i/MiroFish-local`, a local-first fork that adds `ZEP_BACKEND=graphiti` and replaces Zep with locally running Graphiti + Neo4j. It is not the upstream Docker image.

Colab is still temporary: a runtime reset destroys all local data and URLs. Do not use this as production hosting, and do not upload secrets/confidential documents to an unprotected tunnel.

## What was fixed in this revision

- **Driver-aware CUDA wheel selection.** vLLM's default PyPI wheel is CUDA 13-only since v0.20 — mixing it with cu12 torch causes `ImportError: libcudart.so.13`, and installing it over Colab's system Python causes the torch/torchaudio CUDA mismatch crash. Step 3 reads the driver's max CUDA from `nvidia-smi` and installs the matching official per-CUDA vLLM wheels (`wheels.vllm.ai/<version>/cuXXX`) into an isolated uv venv at `/content/modelenv`, leaving Colab's site-packages untouched. A flavor marker rebuilds the venv automatically if the target CUDA changes. `wrapt` is installed explicitly because a vLLM 0.28 dependency ships a `sitecustomize` hook that imports it without declaring it.
- **Neo4j with a three-level fallback.** Colab's gVisor sandbox blocks netfilter NAT, bind mounts, and `unshare` — no container runtime can work. Step 6 tries stock dockerd → vfs/no-iptables/host-network dockerd → a containerless Neo4j (OpenJDK + official tarball). All modes bind Bolt/HTTP to loopback; the backend always sees `bolt://127.0.0.1:7687`. The APOC plugin download retries on transient errors.
- **OASIS installed into the correct venv with an mcp 1.x pin.** After `uv sync --extra graphiti`, `camel-oasis` is force-installed with `uv pip install --python backend/.venv/bin/python` (a bare `uv pip install` can target the system Python), and `mcp>=1.28,<2` is pinned because camel-ai 0.2.78 uses the v1 `FastMCP` import that MCP Python SDK 2.0 removed.
- **Backend start bypasses `uv run`.** `uv run python run.py` re-syncs the project venv to the locked dependency set, silently removing the manual camel-oasis/mcp installs. Step 9 starts `backend/.venv/bin/python run.py` directly with `VIRTUAL_ENV`/`PATH` set.
- **Vite bound to IPv4 explicitly.** The fork's dev script is bare `vite`, and on this Node/Vite 7 stack `localhost` resolves to IPv6 `::1` only — the server looked "ready" while nothing could reach it over IPv4. Step 10 runs `frontend/node_modules/.bin/vite --host 127.0.0.1 --port 3000 --strictPort` directly.
- **Router gates on upstream health and fails with 502, not 500.** The proxy turns unreachable upstreams into a JSON `502` naming the dead service, and the router cell refuses to pass while `llm` or `embed` is not 200.
- **Fork bugfix: lazily started Graphiti event loop.** The fork's `zep_graphiti_impl._run_async` submits coroutines to a module-level loop that is `None` on the graph-build path (`AttributeError: 'NoneType' object has no attribute 'call_soon_threadsafe'` right after a successful ontology generation). Step 8b patches `_run_async` to create and start the loop on first use (idempotent, syntax-checked, original backed up).
- **Ontology-400 diagnosis + narrowly scoped patch.** The endpoint intermittently returns 400 within ~1s with no LLM call — including for the fork's own demo seed. Step 11a replays the request against the local backend and prints the full response body plus the handler's validation code. Step 8c first prints that same diagnosis (read-only, always runs), then applies a patch that only activates once 11a's output identifies the real guard — until then it is a no-op and never masks errors.
- **Tunnels deferred to the end.** Backend and frontend start on loopback first; the pipeline is proven with a headless ontology call; only then does the final "Go public" cell create both Cloudflare quick tunnels. Earlier revisions tunnelled the backend at start time, so a backend restart silently invalidated the frontend's API URL.
- **Liveness checks use `$!` + `kill -0`, not cmdline pgrep.** Between `nohup` exec and a server printing anything there is a window where no process matches a cmdline pattern yet — a pgrep in that window produced false "process exited early" failures while servers were starting normally.
- **Step 7 is idempotent and race-proof.** A healthy existing install (graphiti-core + oasis + mcp 1.x + frontend deps) is reused instead of rebuilt. A stale tree is never `rm -rf`'d while a process may be writing to it (the `Directory not empty` race): the cell kills tree-anchored processes, then moves the old tree aside with an atomic `mv` before cloning.
- **Session-state cell + guards.** Colab VMs are ephemeral and a reconnect (or a fresh upload of this file) means a new, empty VM. The "Session state check" cell near the top prints what survived and which step to resume from; the config and frontend cells fail with clear "run step X first" messages instead of cryptic missing-file errors.
- **Fail-fast + live progress in every long cell.** Cells timestamp each phase, stream the last log line while waiting, and abort immediately with a log tail if a server process dies.
- **No more cosmetic SIGPIPE failures.** `curl ... | head` under `set -o pipefail` exits 23 when head closes the pipe early; health checks now write to temp files before truncating output.
- **vLLM startup flags reduced to what is verified** (`float16` on the cc7.5 T4, `--enforce-eager` for LFM2's hybrid conv blocks, 16K context).
- **Embeddings**: `bge-small-en-v1.5` (384 dims) replaced by `bge-m3` (1024 dims) — graphiti-core defaults to `embedding_dim=1024` and Neo4j vector indexes are fixed-dimension at creation, so 384-dim vectors fail at graph-write time. bge-m3 is also multilingual (EN/PT/ES/ZH).

## 1. Enable a GPU

In Colab choose **Runtime → Change runtime type → T4 GPU**, reconnect, then run this cell. A GPU is required for usable LLM latency. No credentials are requested anywhere in this notebook.

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('No NVIDIA GPU. Enable a Colab T4 GPU, reconnect, then rerun.')
print('Detected:', r.stdout.strip())
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout.splitlines()[2])

## 2. Optional: Hugging Face token

`LiquidAI/LFM2.5-350M` is distributed under the LFM Open License. If Liquid AI enables access gating on the repo, anonymous downloads fail with a 401 and the vLLM cell will say so. In that case:

1. Open [the model page](https://huggingface.co/LiquidAI/LFM2.5-350M) and accept the license.
2. Create a **Read** token at [Hugging Face settings](https://huggingface.co/settings/tokens).
3. Add it to Colab as a secret named `HF_TOKEN` (key icon in the left sidebar) and enable notebook access.

The cell below picks the secret up automatically. If the repo is not gated, everything works without it.

In [ ]:
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('HF_TOKEN loaded from Colab secrets.')
    else:
        print('No HF_TOKEN secret - continuing anonymously (fine while the repo is not gated).')
except Exception:
    print('Colab secrets unavailable - continuing anonymously.')

## Session state check — run this first when resuming

Everything this notebook builds lives under `/content`, which is wiped when Colab assigns a new VM (idle reclaim, runtime reset, or opening a freshly uploaded copy of this file). If a cell fails with "No such file or directory" for `/content/MiroFish-local`, `/content/modelenv` or `/content/neo4j`, run this cell: it prints which artifacts exist, which services answer, and the earliest step to resume from. All setup cells are idempotent, so re-running a step is always safe.

In [ ]:
import os, socket

ARTIFACTS = [
    (3, 'model venv', '/content/modelenv/bin/python'),
    (6, 'host Neo4j install', '/content/neo4j/bin/neo4j'),
    (7, 'repo clone', '/content/MiroFish-local'),
    (7, 'backend venv', '/content/MiroFish-local/backend/.venv/bin/python'),
    (8, '.env config', '/content/MiroFish-local/.env'),
    (13, 'backend tunnel URL', '/content/local-mirofish/backend_url.txt'),
]
print('Artifacts:')
missing_steps = []
for step, label, p in ARTIFACTS:
    ok = os.path.exists(p)
    if not ok:
        missing_steps.append(step)
    print(('  OK     ' if ok else '  MISSING'), f'step {step}: {label}')

def up(port):
    s = socket.socket()
    s.settimeout(0.4)
    try:
        s.connect(('127.0.0.1', port))
        return True
    except OSError:
        return False
    finally:
        s.close()

PORTS = [(8000, 'vLLM'), (8001, 'embeddings'), (9000, 'router'),
         (7687, 'Neo4j bolt'), (5001, 'backend'), (3000, 'frontend')]
print('\nServices:')
for port, name in PORTS:
    print(('  UP     ' if up(port) else '  down  '), f'{port}  {name}')

if missing_steps:
    print('\nResume from step', min(missing_steps))
else:
    print('\nAll artifacts present. For any service marked down, re-run its step.')

## 3. Local LLM: LFM2.5-350M via vLLM (isolated, driver-matched venv)

vLLM ≥ 0.23 supports LFM2.5 natively (`Lfm2ForCausalLM`). It exposes an OpenAI-compatible API only on `127.0.0.1:8000`; it is never tunneled publicly.

**Why this cell is structured this way:** since v0.20, vLLM's default PyPI wheel is compiled for CUDA 13 only, and its native extension then demands `libcudart.so.13`. Installing it over Colab's system Python additionally collides with the preinstalled CUDA-12.8 torchaudio. vLLM publishes per-release, per-CUDA wheels at `wheels.vllm.ai/<version>/cuXXX`, so this cell:

1. Reads the driver's maximum CUDA version from `nvidia-smi`.
2. Driver ≥ 13.0 → `vllm==0.28.0` (cu130 wheels). Driver 12.9 → `vllm==0.23.0` cu129 wheels (0.23 is the first release with LFM2.5 support). Driver ≤ 12.8 → cu128 attempt with the same pin.
3. Installs into `/content/modelenv` (uv venv) together with sentence-transformers/fastapi/uvicorn/httpx, so torch, torchvision and torchaudio always come from the same CUDA index as vLLM. Colab's own packages are untouched. `wrapt` is included because a vLLM dependency's `sitecustomize` hook imports it without declaring it — without this, every venv process prints a harmless but noisy `Error in sitecustomize`.
4. Verifies the stack by actually importing torch, torchaudio and vLLM and asserting `torch.cuda.is_available()` — before spending minutes on a model download.

Server flags for the T4 stay minimal: `--dtype float16` (cc7.5 has no native bf16), `--enforce-eager` (skips fragile CUDA-graph capture on LFM2's hybrid conv blocks), `--max-model-len 16384` (headroom for Graphiti prompts; the 350M KV cache is tiny).

Expect 4-8 min: venv + wheels (~3 GB), then ~0.7 GB of model weights and engine load. Progress streams below; if the server dies (OOM, gated model, driver issue) the cell stops immediately and prints the log.

In [ ]:
%%bash
set -euo pipefail
VENV=/content/modelenv
DRIVER_CUDA=$(nvidia-smi | grep -oE 'CUDA Version: [0-9]+\.[0-9]+' | grep -oE '[0-9]+\.[0-9]+' | head -1 || true)
[ -n "${DRIVER_CUDA:-}" ] || { echo 'Could not parse driver CUDA version from nvidia-smi'; exit 1; }
DMAJ=${DRIVER_CUDA%%.*}
DMIN=${DRIVER_CUDA#*.}
if [ "$DMAJ" -ge 13 ]; then
  FLAVOR=cu130; VLLM_VER=0.28.0; IDX_VLLM=https://wheels.vllm.ai/0.28.0/cu130; IDX_TORCH=https://download.pytorch.org/whl/cu130
elif [ "$DMAJ" -eq 12 ] && [ "$DMIN" -ge 9 ]; then
  FLAVOR=cu129; VLLM_VER=0.23.0; IDX_VLLM=https://wheels.vllm.ai/0.23.0/cu129; IDX_TORCH=https://download.pytorch.org/whl/cu129
else
  FLAVOR=cu128; VLLM_VER=0.23.0; IDX_VLLM=https://wheels.vllm.ai/0.23.0/cu128; IDX_TORCH=https://download.pytorch.org/whl/cu128
  echo 'NOTE: driver reports CUDA <= 12.8. vLLM 0.23 mainly publishes cu129/cu130 wheels; if the cu128 index is unavailable this cell will fail - report it and consider a runtime with a newer driver.'
fi
echo "[$(date +%H:%M:%S)] Driver supports CUDA $DRIVER_CUDA -> $FLAVOR wheels, vllm==$VLLM_VER"
pip -q install -U uv
if [ -d "$VENV" ] && [ ! -f "$VENV/.flavor" ]; then rm -rf "$VENV"; fi
if [ -f "$VENV/.flavor" ] && [ "$(cat "$VENV/.flavor")" != "$FLAVOR" ]; then
  echo "[$(date +%H:%M:%S)] CUDA flavor changed ($(cat "$VENV/.flavor") -> $FLAVOR) - recreating venv"
  rm -rf "$VENV"
fi
[ -x "$VENV/bin/python" ] || { echo "[$(date +%H:%M:%S)] Creating model venv (python 3.12)..."; uv venv "$VENV" --python 3.12; }
echo "[$(date +%H:%M:%S)] Installing vllm==$VLLM_VER + sentence-transformers ($FLAVOR wheels; 3-6 min)..."
uv pip install --python "$VENV/bin/python" \
  "vllm==$VLLM_VER" sentence-transformers fastapi uvicorn httpx wrapt \
  --extra-index-url "$IDX_VLLM" --extra-index-url "$IDX_TORCH" \
  --index-strategy unsafe-best-match
echo "$FLAVOR" > "$VENV/.flavor"
echo "[$(date +%H:%M:%S)] Stack consistency check:"
"$VENV/bin/python" - <<'PY'
import torch, torchaudio, vllm
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| torchaudio', torchaudio.__version__, '| vllm', vllm.__version__)
if not torch.cuda.is_available():
    raise SystemExit('CUDA not visible to torch inside the venv - driver too old for the selected wheels; see step 3 notes.')
print('device:', torch.cuda.get_device_name(0), '| compute capability:', torch.cuda.get_device_capability(0))
PY
mkdir -p /content/local-mirofish/logs
pkill -f 'vllm.entrypoints.openai.api_server' 2>/dev/null || true
sleep 3
nohup "$VENV/bin/python" -m vllm.entrypoints.openai.api_server \
  --model LiquidAI/LFM2.5-350M \
  --served-model-name LiquidAI/LFM2.5-350M \
  --dtype float16 \
  --max-model-len 32768 \
  --gpu-memory-utilization 0.65 \
  --enforce-eager \
  --host 127.0.0.1 --port 8000 \
  > /content/local-mirofish/logs/vllm.log 2>&1 &
VLLM_PID=$!
echo "[$(date +%H:%M:%S)] vLLM pid $VLLM_PID - first start downloads ~0.7 GB of weights, then loads the engine (2-6 min)."
for i in $(seq 1 240); do
  if curl -fsS --max-time 3 http://127.0.0.1:8000/v1/models >/dev/null 2>&1; then
    echo "[$(date +%H:%M:%S)] vLLM ready:"
    curl -fsS http://127.0.0.1:8000/v1/models
    exit 0
  fi
  if ! kill -0 "$VLLM_PID" 2>/dev/null; then
    echo "[$(date +%H:%M:%S)] vLLM process exited early - log tail:"
    tail -n 120 /content/local-mirofish/logs/vllm.log
    if grep -Eiq 'gated|401|unauthorized|authentication' /content/local-mirofish/logs/vllm.log; then
      echo '---'
      echo 'Model download was refused: the LiquidAI repo appears to be gated.'
      echo 'Accept the license on the model page, add a Read token as Colab secret HF_TOKEN,'
      echo 'then re-run cell 2 and this cell.'
    fi
    exit 1
  fi
  if [ $((i % 10)) -eq 0 ]; then
    echo "[$(date +%H:%M:%S)] still starting ($((i*2))s): $(tail -n 1 /content/local-mirofish/logs/vllm.log | cut -c1-200)"
  fi
  sleep 2
done
echo 'Timed out after 8 minutes - log tail:'
tail -n 120 /content/local-mirofish/logs/vllm.log
exit 1

## 4. Local embedding API

Graphiti needs an OpenAI-compatible `/v1/embeddings` endpoint as well as a chat endpoint. This small FastAPI service loads `BAAI/bge-m3` on the same GPU and runs from the isolated `/content/modelenv` venv created in step 3 (running it on Colab's system Python would hit the same torch/torchaudio mismatch through transformers).

Why bge-m3 and not bge-small: graphiti-core's embedder config defaults to `embedding_dim=1024` and truncates/pads to that size, while Neo4j vector indexes are created with a fixed dimension and reject mismatched vectors at write time. A 384-dim model therefore breaks graph builds. bge-m3 emits exactly 1024 dims, is multilingual (EN/PT/ES/ZH — useful for non-English seed documents), and still fits easily on the T4 next to vLLM. The server additionally pads/truncates every vector to 1024 as a safety net; zero-padding does not change cosine similarity, which is what the Neo4j index uses.

In [ ]:
%%bash
set -euo pipefail
VENV=/content/modelenv
[ -x "$VENV/bin/python" ] || { echo 'model venv missing - run the vLLM cell first'; exit 1; }
cat > /content/local-mirofish/embed_server.py <<'PY'
import os
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'BAAI/bge-m3'
TARGET_DIM = int(os.environ.get('EMBED_DIM', '1024'))  # graphiti-core default embedding_dim
model = SentenceTransformer(MODEL_NAME, device='cuda')
app = FastAPI()

class EmbeddingRequest(BaseModel):
    input: str | list[str]
    model: str | None = None
    encoding_format: str | None = None

def fit_dim(v):
    if len(v) >= TARGET_DIM:
        return v[:TARGET_DIM]
    return v + [0.0] * (TARGET_DIM - len(v))  # zero-padding preserves cosine similarity

@app.get('/health')
def health():
    return {'ok': True, 'model': MODEL_NAME, 'dim': TARGET_DIM}

@app.post('/v1/embeddings')
def embeddings(req: EmbeddingRequest):
    texts = [req.input] if isinstance(req.input, str) else req.input
    vectors = model.encode(texts, normalize_embeddings=True).tolist()
    data = [{'object': 'embedding', 'embedding': fit_dim(v), 'index': i} for i, v in enumerate(vectors)]
    return {'object': 'list', 'data': data, 'model': MODEL_NAME,
            'usage': {'prompt_tokens': 0, 'total_tokens': 0}}
PY
pkill -f 'uvicorn.*embed_server' 2>/dev/null || true
sleep 2
nohup "$VENV/bin/uvicorn" --app-dir /content/local-mirofish embed_server:app --host 127.0.0.1 --port 8001 > /content/local-mirofish/logs/embed.log 2>&1 &
EMB_PID=$!
echo "[$(date +%H:%M:%S)] Embedding server pid $EMB_PID - first start downloads ~2.3 GB (bge-m3 weights)."
ok=''
for i in $(seq 1 150); do
  if curl -fsS --max-time 3 http://127.0.0.1:8001/health >/dev/null 2>&1; then ok=1; break; fi
  if ! kill -0 "$EMB_PID" 2>/dev/null; then
    echo 'Embedding server exited early - log tail:'
    tail -n 60 /content/local-mirofish/logs/embed.log
    exit 1
  fi
  if [ $((i % 15)) -eq 0 ]; then
    echo "[$(date +%H:%M:%S)] still loading ($((i*2))s): $(tail -n 1 /content/local-mirofish/logs/embed.log | cut -c1-200)"
  fi
  sleep 2
done
if [ -z "$ok" ]; then tail -n 60 /content/local-mirofish/logs/embed.log; exit 1; fi
echo "[$(date +%H:%M:%S)] Embedding server ready:"
curl -fsS http://127.0.0.1:8001/health
echo

### The OpenAI router (one base URL for both services)

The fork maps Graphiti and app LLM settings to one OpenAI-style base URL. This router splits traffic: `v1/embeddings` → the bge-m3 service (8001), everything else → vLLM (8000). Upstreams that are unreachable now produce a JSON **502 naming the dead service** instead of an unhandled 500, and the cell refuses to pass while either upstream is down — with remediation, not a cryptic curl error.

In [ ]:
%%bash
set -euo pipefail
VENV=/content/modelenv
[ -x "$VENV/bin/python" ] || { echo 'model venv missing - run the vLLM cell first'; exit 1; }
cat > /content/local-mirofish/openai_router.py <<'PY'
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse, JSONResponse
from starlette.background import BackgroundTask
import httpx

EMBED = 'http://127.0.0.1:8001'
LLM = 'http://127.0.0.1:8000'
HOP_BY_HOP = {'host', 'content-length', 'connection', 'accept-encoding', 'transfer-encoding'}
app = FastAPI()
client = httpx.AsyncClient(timeout=httpx.Timeout(600.0, connect=10.0))

@app.get('/health')
async def health():
    out = {}
    for name, base in (('llm', LLM), ('embed', EMBED)):
        try:
            r = await client.get(base + '/v1/models' if name == 'llm' else base + '/health', timeout=5)
            out[name] = r.status_code
        except Exception as e:
            out[name] = f'down: {e}'
    return out

@app.api_route('/{path:path}', methods=['GET', 'POST', 'PUT', 'PATCH', 'DELETE'])
async def route(path: str, request: Request):
    target = EMBED if path == 'v1/embeddings' else LLM
    headers = {k: v for k, v in request.headers.items() if k.lower() not in HOP_BY_HOP}
    req = client.build_request(request.method, f'{target}/{path}',
                               params=request.query_params,
                               content=await request.body(), headers=headers)
    try:
        upstream = await client.send(req, stream=True)
    except httpx.HTTPError as e:
        which = 'embeddings service on :8001' if target == EMBED else 'vLLM on :8000'
        return JSONResponse({'error': f'upstream {which} unreachable: {e}',
                             'hint': 'vLLM down -> re-run step 3; embeddings down -> re-run step 4'}, status_code=502)
    out_headers = {k: v for k, v in upstream.headers.items() if k.lower() not in HOP_BY_HOP}
    return StreamingResponse(upstream.aiter_raw(), status_code=upstream.status_code,
                             headers=out_headers, background=BackgroundTask(upstream.aclose))
PY
pkill -f 'uvicorn.*openai_router' 2>/dev/null || true
sleep 2
nohup "$VENV/bin/uvicorn" --app-dir /content/local-mirofish openai_router:app --host 127.0.0.1 --port 9000 > /content/local-mirofish/logs/router.log 2>&1 &
ROUTER_PID=$!
ready=''
for i in $(seq 1 30); do
  if curl -fsS --max-time 3 http://127.0.0.1:9000/health >/dev/null 2>&1; then ready=1; break; fi
  kill -0 "$ROUTER_PID" 2>/dev/null || { echo 'Router exited early:'; tail -n 40 /content/local-mirofish/logs/router.log; exit 1; }
  sleep 2
done
[ -n "$ready" ] || { echo 'Router did not become ready:'; tail -n 40 /content/local-mirofish/logs/router.log; exit 1; }

echo 'Router health:'
curl -fsS http://127.0.0.1:9000/health -o /tmp/router_health.json
cat /tmp/router_health.json; echo
if ! grep -q '"llm":200' /tmp/router_health.json; then
  echo '---'
  echo 'vLLM is NOT answering on 127.0.0.1:8000 - the router is up but the LLM upstream is dead.'
  echo 'Check:  pgrep -af vllm.entrypoints ; tail -n 30 /content/local-mirofish/logs/vllm.log ; free -g'
  echo 'Fix:    re-run step 3 (fast when the venv and weights exist), then re-run this cell.'
  exit 1
fi
if ! grep -q '"embed":200' /tmp/router_health.json; then
  echo '---'
  echo 'Embedding service is NOT answering on 127.0.0.1:8001 - re-run the embedding cell in step 4.'
  exit 1
fi
curl -fsS http://127.0.0.1:9000/v1/models -o /tmp/router_models.json || { echo 'models probe failed:'; cat /tmp/router_models.json 2>/dev/null; exit 1; }
cat /tmp/router_models.json; echo
curl -fsS http://127.0.0.1:9000/v1/embeddings -H 'content-type: application/json' -d '{"input":"local graph embedding test","model":"BAAI/bge-m3"}' -o /tmp/router_emb.json || { echo 'embeddings probe failed:'; cat /tmp/router_emb.json 2>/dev/null; exit 1; }
echo 'Embedding response (first 200 bytes):'
head -c 200 /tmp/router_emb.json; echo

## 5. Verify a completely local model response

This confirms the routing layer serves the exact OpenAI-chat protocol MiroFish uses. The request never leaves this Colab runtime. Sampling follows the LFM2.5 recipe (temperature 0.1, `top_k` 50, `repetition_penalty` 1.05) — these are per-request extras, not server flags.

In [ ]:
import requests
payload = {
    'model': 'LiquidAI/LFM2.5-350M',
    'messages': [
        {'role': 'system', 'content': 'Reply only with JSON.'},
        {'role': 'user', 'content': 'Return {"local": true}'},
    ],
    'temperature': 0.1,
    'top_k': 50,
    'repetition_penalty': 1.05,
    'max_tokens': 32,
    'response_format': {'type': 'json_object'},
}
response = requests.post('http://127.0.0.1:9000/v1/chat/completions', json=payload, timeout=120)
if response.status_code == 502:
    raise RuntimeError(f"Router reports a dead upstream: {response.json()}")
response.raise_for_status()
print(response.json()['choices'][0]['message']['content'])

## 6. Docker and local Neo4j (with containerless fallback)

Neo4j holds the Graphiti knowledge graph. The password below is VM-local and temporary; it is not a production credential.

**Why not just Docker:** Colab sandboxes the VM with gVisor, which blocks every primitive a container runtime needs: netfilter NAT (`failed to create NAT chain DOCKER`), bind mounts (the containerd snapshotter cannot extract layers), and the `unshare` syscall (`failed to register layer: unshare: operation not permitted`). No flag or storage driver fixes this — rootless Podman, nerdctl, bubblewrap, Apptainer and gVisor-based runtimes all need the same blocked syscalls. If a future dependency exists only as an image, the working escape hatches are: flatten it with `crane export <image> - | tar -x` and run from the rootfs (optionally under `proot -R`), or use udocker (PRoot engine) — both are pure user-space and need no namespaces. For Neo4j neither is needed: it is a plain JVM application.

**What this cell does** — tries three modes in order and keeps the first that answers a Bolt `RETURN 1`:

1. **bridge** — stock dockerd with loopback-published ports (works on normal machines/VMs).
2. **host** — dockerd with a daemon.json that disables iptables, the bridge, and the containerd snapshotter, using the `vfs` storage driver; Neo4j with `--network host`, loopback-pinned.
3. **hostpkg** — no Docker at all: OpenJDK 17 + the official `neo4j-community` tarball from dist.neo4j.org, loopback-only, capped heap/pagecache. This is the mode that works on Colab. APOC core is added best-effort with retries (Graphiti does not require it).

All three give the backend the same thing: `bolt://127.0.0.1:7687`, user `neo4j`, password `password`. The winning mode is saved to `docker_mode.txt`; re-running the cell first checks whether Neo4j already answers and exits immediately if so.

In [ ]:
%%bash
set -euo pipefail
export DEBIAN_FRONTEND=noninteractive
LOGS=/content/local-mirofish/logs
MODE_FILE=/content/local-mirofish/docker_mode.txt
mkdir -p "$LOGS"

if docker exec mirofish-neo4j cypher-shell -u neo4j -p password 'RETURN 1' >/dev/null 2>&1; then
  echo 'Neo4j already running (docker) - nothing to do'; exit 0
fi
if [ -x /content/neo4j/bin/cypher-shell ] && /content/neo4j/bin/cypher-shell -u neo4j -p password 'RETURN 1' >/dev/null 2>&1; then
  echo 'Neo4j already running (host install) - nothing to do'; exit 0
fi

echo "[$(date +%H:%M:%S)] Installing docker.io..."
apt-get -qq update
apt-get -qq install -y docker.io git curl ca-certificates
update-alternatives --set iptables /usr/sbin/iptables-nft 2>/dev/null || true
update-alternatives --set ip6tables /usr/sbin/ip6tables-nft 2>/dev/null || true

start_dockerd() {
  pkill -x dockerd 2>/dev/null || true
  sleep 2
  nohup dockerd "$@" > "$LOGS/dockerd.log" 2>&1 &
  DPID=$!
  for i in $(seq 1 45); do
    docker info >/dev/null 2>&1 && return 0
    kill -0 "$DPID" 2>/dev/null || return 1
    sleep 2
  done
  return 1
}

write_vfs_daemon_json() {
  mkdir -p /etc/docker
  printf '%s
' '{' '  "iptables": false,' '  "ip6tables": false,' '  "bridge": "none",' '  "storage-driver": "vfs",' '  "features": {"containerd-snapshotter": false}' '}' > /etc/docker/daemon.json
}

try_docker_neo4j() {
  local NETMODE=$1
  docker rm -f mirofish-neo4j >/dev/null 2>&1 || true
  docker volume rm mirofish_neo4j_data >/dev/null 2>&1 || true
  if [ "$NETMODE" = host ]; then
    docker run -d --name mirofish-neo4j --restart=no \
      --network host \
      -e NEO4J_AUTH=neo4j/password \
      -e NEO4J_PLUGINS='["apoc"]' \
      -e NEO4J_server_default__listen__address=127.0.0.1 \
      -e NEO4J_server_memory_heap_max__size=1500m \
      -e NEO4J_server_memory_pagecache_size=512m \
      -v mirofish_neo4j_data:/data \
      neo4j:5.26 || return 1
  else
    docker run -d --name mirofish-neo4j --restart=no \
      -p 127.0.0.1:7474:7474 -p 127.0.0.1:7687:7687 \
      -e NEO4J_AUTH=neo4j/password \
      -e NEO4J_PLUGINS='["apoc"]' \
      -e NEO4J_server_memory_heap_max__size=1500m \
      -e NEO4J_server_memory_pagecache_size=512m \
      -v mirofish_neo4j_data:/data \
      neo4j:5.26 || return 1
  fi
  echo "[$(date +%H:%M:%S)] Container started ($NETMODE); first pull is ~1 GB..."
  for i in $(seq 1 90); do
    if docker exec mirofish-neo4j cypher-shell -u neo4j -p password 'RETURN 1' >/dev/null 2>&1; then
      return 0
    fi
    local R
    R=$(docker inspect -f '{{.State.Running}}' mirofish-neo4j 2>/dev/null || echo false)
    if [ "$R" != true ]; then
      echo 'Container exited early - log tail:'
      docker logs --tail 40 mirofish-neo4j 2>&1 || true
      return 1
    fi
    if [ $((i % 15)) -eq 0 ]; then echo "[$(date +%H:%M:%S)] waiting for Neo4j ($((i*2))s)..."; fi
    sleep 2
  done
  docker logs --tail 60 mirofish-neo4j 2>&1 || true
  return 1
}

install_host_neo4j() {
  echo "[$(date +%H:%M:%S)] Installing containerless Neo4j (OpenJDK 17 + official tarball)..."
  command -v java >/dev/null 2>&1 || apt-get -qq install -y openjdk-17-jre-headless
  local NEO4J_VER=5.26.0
  if [ ! -x /content/neo4j/bin/neo4j ]; then
    curl -fsSL -o /tmp/neo4j.tgz "https://dist.neo4j.org/neo4j-community-${NEO4J_VER}-unix.tar.gz"
    rm -rf /content/neo4j
    mkdir -p /content/neo4j
    tar -xzf /tmp/neo4j.tgz -C /content/neo4j --strip-components=1
  fi
  sed -i '/^server.default_listen_address=/d;/^server.memory.heap.max_size=/d;/^server.memory.pagecache.size=/d' /content/neo4j/conf/neo4j.conf
  printf '%s
' 'server.default_listen_address=127.0.0.1' 'server.memory.heap.max_size=1500m' 'server.memory.pagecache.size=512m' >> /content/neo4j/conf/neo4j.conf
  if [ ! -f /content/neo4j/plugins/apoc-${NEO4J_VER}-core.jar ]; then
    local ok='' attempt
    for attempt in 1 2 3; do
      if curl -fsSL --retry 2 -o /tmp/apoc.jar "https://github.com/neo4j/apoc/releases/download/${NEO4J_VER}/apoc-${NEO4J_VER}-core.jar"; then
        mv /tmp/apoc.jar "/content/neo4j/plugins/apoc-${NEO4J_VER}-core.jar"
        echo 'APOC core installed'
        ok=1
        break
      fi
      echo "APOC download attempt $attempt failed (transient GitHub 5xx is common) - retrying in 5s..."
      sleep 5
    done
    [ -n "$ok" ] || echo 'APOC unavailable - continuing without it (Graphiti does not require APOC)'
  fi
  /content/neo4j/bin/neo4j-admin dbms set-initial-password password >/dev/null 2>&1 || true
  pkill -f '/content/neo4j' 2>/dev/null || true
  sleep 2
  nohup /content/neo4j/bin/neo4j console > "$LOGS/neo4j.log" 2>&1 &
}

NETMODE=$(cat "$MODE_FILE" 2>/dev/null || echo bridge)
[ "$NETMODE" = hostpkg ] && NETMODE=bridge
if ! docker info >/dev/null 2>&1; then
  rm -f /etc/docker/daemon.json
  echo "[$(date +%H:%M:%S)] Starting dockerd (default networking)..."
  if start_dockerd; then
    NETMODE=bridge
  else
    echo "[$(date +%H:%M:%S)] Stock dockerd failed (kernel blocks netfilter NAT - normal on Colab):"
    tail -n 6 "$LOGS/dockerd.log"
    echo "[$(date +%H:%M:%S)] Retrying dockerd with vfs storage + no iptables + no containerd snapshotter..."
    write_vfs_daemon_json
    if start_dockerd; then NETMODE=host; else NETMODE=none; fi
  fi
fi

SUCCESS=''
if [ "$NETMODE" != none ]; then
  if try_docker_neo4j "$NETMODE"; then
    SUCCESS=1
  else
    echo "[$(date +%H:%M:%S)] Docker run failed (image extract or container start) - reconfiguring daemon to vfs/no-iptables and retrying once..."
    write_vfs_daemon_json
    if start_dockerd && try_docker_neo4j host; then
      NETMODE=host
      SUCCESS=1
    else
      NETMODE=none
    fi
  fi
fi

if [ -n "$SUCCESS" ]; then
  echo "$NETMODE" > "$MODE_FILE"
  echo "[$(date +%H:%M:%S)] Neo4j ready (docker, mode: $NETMODE)"
  exit 0
fi

echo "[$(date +%H:%M:%S)] Docker cannot run containers here - switching to containerless Neo4j."
install_host_neo4j
for i in $(seq 1 90); do
  if /content/neo4j/bin/cypher-shell -u neo4j -p password 'RETURN 1' >/dev/null 2>&1; then
    echo 'hostpkg' > "$MODE_FILE"
    echo "[$(date +%H:%M:%S)] Neo4j ready (containerless host install, bolt on 127.0.0.1:7687)"
    exit 0
  fi
  if ! pgrep -f '/content/neo4j' >/dev/null 2>&1; then
    echo 'Host Neo4j process died - log tail:'
    tail -n 60 "$LOGS/neo4j.log"
    exit 1
  fi
  if [ $((i % 15)) -eq 0 ]; then echo "[$(date +%H:%M:%S)] waiting for host Neo4j ($((i*2))s)..."; fi
  sleep 2
done
tail -n 80 "$LOGS/neo4j.log"
exit 1

## 7. Get the local-first MiroFish fork and install it

This fork is required: the upstream MiroFish backend hard-depends on Zep Cloud. The install can take several minutes.

**Idempotency first:** if a previous run already produced a healthy install (backend venv imports `graphiti_core` + `oasis` + `neo4j`, mcp is 1.x, frontend deps present), the cell reuses it and exits in seconds. Otherwise it stops any process anchored to the old tree and moves it aside with an atomic `mv` — never `rm -rf` on a live tree, which loses the race with a still-writing process (`rm: cannot remove ...: Directory not empty`).

Three fork/dependency traps are handled on a fresh install:

1. **Conflicting extras.** The fork declares `graphiti` and `oasis` mutually exclusive (issue `tt-a1i/MiroFish-local#3`: camel-oasis pins an older `neo4j` driver). So after `uv sync --extra graphiti`, the cell installs `camel-oasis` on top with the pip-style resolver.
2. **Correct venv targeting.** A bare `uv pip install camel-oasis` can target Colab's system Python (`/usr`) instead of the backend project. Every install/verify here goes through the explicit interpreter `backend/.venv/bin/python`.
3. **mcp 2.x breakage.** camel-ai 0.2.78 uses the MCP v1 import `from mcp.server import FastMCP`; MCP Python SDK 2.0 removed it. The cell pins `mcp>=1.28,<2` inside the backend venv.

The expected `neo4j` driver downgrade to 5.23.0 inside the backend venv is camel-oasis's pin; it still works with the Neo4j 5.26 server. After this cell, never run `uv sync`/`uv run` in the backend again — a re-sync prunes the manually added packages (step 9 starts the backend with the venv interpreter directly for this reason). To force a clean reinstall instead of reuse, stop services and delete `/content/MiroFish-local` first.

In [ ]:
%%bash
set -euo pipefail
export DEBIAN_FRONTEND=noninteractive
ROOT=/content/MiroFish-local
BACKEND="$ROOT/backend"
VENV="$BACKEND/.venv"

echo "[$(date +%H:%M:%S)] Checking for a healthy existing install..."
if [ -x "$VENV/bin/python" ] && [ -d "$ROOT/frontend/node_modules" ] && "$VENV/bin/python" - >/dev/null 2>&1 <<'PY'
from importlib.metadata import version
import graphiti_core, oasis, neo4j
assert int(version('mcp').split('.')[0]) == 1, 'mcp 2.x installed'
PY
then
  echo 'Existing install is complete (graphiti-core + oasis + mcp 1.x + frontend deps) - reusing it, nothing to do.'
  echo 'To force a clean reinstall, delete /content/MiroFish-local first.'
  exit 0
fi

echo "[$(date +%H:%M:%S)] Installing Node.js 20, Git, curl, and uv..."
apt-get -qq update
apt-get -qq install -y git curl ca-certificates
if ! command -v node >/dev/null 2>&1 || ! node --version | grep -q '^v20\.'; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null
  apt-get -qq install -y nodejs
fi
pip -q install -U uv
echo "Node: $(node --version)"
echo "npm:  $(npm --version)"
echo "uv:   $(uv --version)"

echo
echo "[$(date +%H:%M:%S)] Stopping anything anchored to a previous tree..."
pkill -f '/content/MiroFish-local' 2>/dev/null || true
pkill -f 'npm run' 2>/dev/null || true
sleep 2
if [ -e "$ROOT" ]; then
  STALE="${ROOT}.stale.$(date +%s)"
  echo "[$(date +%H:%M:%S)] Moving old tree aside to $STALE (atomic mv - immune to the rm ENOTEMPTY race)..."
  mv "$ROOT" "$STALE"
  (rm -rf "$STALE" >/dev/null 2>&1 &)
fi

echo "[$(date +%H:%M:%S)] Cloning tt-a1i/MiroFish-local..."
git clone --depth 1 https://github.com/tt-a1i/MiroFish-local.git "$ROOT"

echo
echo "[$(date +%H:%M:%S)] Installing root and frontend Node dependencies..."
cd "$ROOT"
npm run setup

echo
echo "[$(date +%H:%M:%S)] Creating/syncing the MiroFish BACKEND virtualenv with Graphiti..."
cd "$BACKEND"
uv sync --extra graphiti

test -x "$VENV/bin/python" || {
  echo "ERROR: Expected backend virtualenv was not created: $VENV"
  exit 1
}

echo
echo "[$(date +%H:%M:%S)] Confirming Graphiti is installed in the backend venv..."
"$VENV/bin/python" - <<'PY'
import sys
import graphiti_core
print("Backend Python:", sys.executable)
print("graphiti-core: OK")
PY

echo
echo "[$(date +%H:%M:%S)] Checking whether OASIS is already installed in the SAME backend venv..."
if "$VENV/bin/python" -c 'import oasis' >/dev/null 2>&1; then
  echo "OASIS already exists in backend .venv."
else
  echo "OASIS is absent after graphiti-only sync."
  echo "Installing camel-oasis explicitly into: $VENV"
  uv pip install --python "$VENV/bin/python" camel-oasis
fi

echo
echo "[$(date +%H:%M:%S)] Pinning mcp to the 1.x line (camel-ai 0.2.78 uses the v1 FastMCP import; MCP SDK 2.0 removed it)..."
uv pip install --python "$VENV/bin/python" "mcp>=1.28,<2"

echo
echo "[$(date +%H:%M:%S)] Final verification: imports from the REAL backend environment..."
"$VENV/bin/python" - <<'PY'
import sys
import neo4j
import graphiti_core
import oasis
print("Backend Python:", sys.executable)
print("neo4j driver:  ", neo4j.__version__)
print("graphiti-core: OK")
print("oasis:         OK")
PY

echo
echo "[$(date +%H:%M:%S)] MiroFish-local backend install completed successfully."
echo "Next: run cell 8 to write the local Graphiti + Neo4j + vLLM configuration."

## 8. Configure Graphiti mode — no Zep variables

`ZEP_BACKEND=graphiti` is the decisive setting. `OPENAI_*` points at the all-local router (the fork would auto-map them from `LLM_*`; they are written explicitly for clarity). `GRAPHITI_EMBEDDING_MODEL` now names the local bge-m3 service, and `GRAPHITI_EMBEDDING_DIM=1024` documents the dimension contract for fork revisions that read it. No `ZEP_API_KEY` is written.

The backend and frontend run on the Colab host, so they access local services at `127.0.0.1`, not `host.docker.internal`.

In [ ]:
%%bash
set -euo pipefail
if [ ! -d /content/MiroFish-local/backend ]; then
  echo 'The fork is not present in this runtime (/content/MiroFish-local is missing).'
  echo 'Run step 7 first - it clones the repo and builds the backend venv.'
  echo 'If you resumed an old session, run the "Session state check" cell near the top to see what else is missing.'
  exit 1
fi
cat > /content/MiroFish-local/.env <<'EOF'
LLM_API_KEY=local
LLM_BASE_URL=http://127.0.0.1:9000/v1
LLM_MODEL_NAME=LiquidAI/LFM2.5-350M
ZEP_BACKEND=graphiti
NEO4J_URI=bolt://127.0.0.1:7687
NEO4J_USER=neo4j
NEO4J_PASSWORD=password
OPENAI_API_KEY=local
OPENAI_BASE_URL=http://127.0.0.1:9000/v1
GRAPHITI_LLM_MODEL=LiquidAI/LFM2.5-350M
GRAPHITI_EMBEDDING_MODEL=BAAI/bge-m3
GRAPHITI_EMBEDDING_DIM=1024
FLASK_DEBUG=false
EOF
grep -E '^(LLM_|ZEP_BACKEND|NEO4J_|OPENAI_|GRAPHITI_|FLASK_DEBUG)' /content/MiroFish-local/.env

## 8b. Patch the fork's Graphiti event loop (required for graph builds)

Confirmed live bug in the fork's current revision: ontology generation succeeds, then the graph build dies at `zep_graphiti_impl.py` with `AttributeError: 'NoneType' object has no attribute 'call_soon_threadsafe'`, and after a naive lazy-loop fix, with `RuntimeError: Future attached to a different loop`.

Graphiti's `AsyncNeo4jDriver` creates sockets/Futures bound to the event loop that first uses the driver, so **every** Graphiti call — initialization, `build_indices_and_constraints`, and `add_episode_bulk` — must run on one persistent dedicated loop thread. A per-call fresh loop fails because the driver's Futures stay bound to the original loop.

This cell replaces `_run_async` with a thread-safe single-loop implementation: it starts the dedicated loop thread if needed, waits for it, and submits work via `run_coroutine_threadsafe`. Idempotent, syntax-checked, original backed up. Restart the backend afterwards (step 9).

In [ ]:
import pathlib, re

TARGET = pathlib.Path(
    "/content/MiroFish-local/backend/app/services/zep_graphiti_impl.py"
)
if not TARGET.exists():
    raise SystemExit(f"Missing file: {TARGET}")

src = TARGET.read_text()

start_match = re.search(r"(?m)^def _run_async\(coro\):\s*$", src)
if not start_match:
    raise SystemExit("Could not find def _run_async(coro):")

after = src[start_match.end():]
end_match = re.search(r"(?m)^(?=def |class )", after)
end_pos = start_match.end() + end_match.start() if end_match else len(src)

old_block = src[start_match.start():end_pos]
print("=== Replacing _run_async ===")
print(old_block[:2500])
print("=== End ===")

NEW_RUN_ASYNC = '''def _run_async(coro):
    """
    Run a Graphiti coroutine on one long-lived dedicated event loop.

    Graphiti's AsyncNeo4jDriver creates sockets/Futures tied to the event loop
    that first uses the driver. Every later Graphiti operation must therefore
    be submitted to that exact same loop. Creating a new loop per call causes:
    RuntimeError: Future attached to a different loop.
    """
    global _async_loop, _async_thread

    if _async_thread is None or not _async_thread.is_alive():
        with _init_lock:
            if _async_thread is None or not _async_thread.is_alive():
                _async_loop = None
                _async_thread = threading.Thread(
                    target=_start_async_loop,
                    daemon=True,
                    name="graphiti-async-loop",
                )
                _async_thread.start()
                for _ in range(500):
                    if _async_loop is not None and _async_loop.is_running():
                        break
                    import time
                    time.sleep(0.01)
                else:
                    raise RuntimeError("Graphiti async event loop did not start within 5 seconds")

    future = asyncio.run_coroutine_threadsafe(coro, _async_loop)
    return future.result(timeout=300)

'''

new_src = src[:start_match.start()] + NEW_RUN_ASYNC + src[end_pos:]

# Remove the bad lazy guard inserted by an earlier patch if present.
bad_lazy_guard = (
    "    global _async_loop\n"
    "    if _async_loop is None or not _async_loop.is_running():\n"
    "        import threading as _threading\n"
    "        _async_loop = asyncio.new_event_loop()\n"
    "        _threading.Thread(target=_async_loop.run_forever, daemon=True).start()\n"
)
new_src = new_src.replace(bad_lazy_guard, "")

compile(new_src, str(TARGET), "exec")
backup = TARGET.with_suffix(".py.before_single_loop_fix.bak")
if not backup.exists():
    backup.write_text(src)
TARGET.write_text(new_src)

print("✓ _run_async now runs every Graphiti call on one persistent loop thread.")
print("✓ Backup:", backup)
print("Restart the backend (step 9) so the patch loads.")


## 8c. Ontology request contract (400 root cause)

The endpoint `/api/graph/ontology/generate` intermittently returned 400 within ~1s with no LLM call. The diagnosis (step 11a probes) proved the cause: the handler requires the multipart form field **`simulation_requirement`** — `{"error":"请提供模拟需求描述 (simulation_requirement)"}` when it is absent. The frontend omitted it; demo.py sent it and succeeded. The cell below is the read-only diagnosis: it replays the request with several field-name variants and prints the full response body plus the handler's validation code, so any future 400 is self-describing.

In [ ]:
%%bash
set +e
ROOT=/content/MiroFish-local
BACK="$ROOT/backend"
if [ ! -d "$BACK/app" ]; then
  echo 'repo not present in THIS VM - run the Session state check first'
  exit 0
fi

echo '=== 1) replay minimal request, print full response body ==='
TMPD=$(mktemp -d)
printf 'A small known-good seed. LocalSDR sells a local AI sales copilot to Spanish SMBs for 29 euros per month. Agency owners care about GDPR and cost.
' > "$TMPD/seed.txt"
if curl -sS --max-time 3 http://127.0.0.1:5001/health >/dev/null 2>&1; then
  curl -sS -o "$TMPD/resp.json" -w 'HTTP %{http_code}
'     -F "files=@$TMPD/seed.txt"     -F "simulation_requirement=Predict how Spanish SMBs react to a local AI sales copilot in the first 60 days."     -F "project_name=debug-ontology-400"     http://127.0.0.1:5001/api/graph/ontology/generate
  echo '--- response body ---'
  cat "$TMPD/resp.json"; echo
else
  echo '(backend not up on 5001 - skipping replay; re-run step 9 first)'
fi

echo
echo '=== 2) handler validation block ==='
f=$(grep -rln "ontology/generate\|generate_ontology\|ontology_generate" "$BACK/app" --include='*.py' | head -1)
echo "HANDLER FILE: $f"
grep -n -B6 -A55 "def .*ontolog" "$f" | head -100

echo
echo '=== 3) backend log lines near recent 400s ==='
grep -nE 'WARNING|ERROR|400|本体|ontology' /content/local-mirofish/logs/mirofish-backend.log 2>/dev/null | tail -20

In [ ]:
%%bash
set +e
ROOT=/content/MiroFish-local
if [ ! -d "$ROOT/backend/app" ]; then
  echo 'repo not present in THIS VM - run the Session state check first'
  exit 0
fi
if ! curl -sS --max-time 3 http://127.0.0.1:5001/health >/dev/null 2>&1; then
  echo 'backend not answering on 5001 - re-run step 9 first'
  exit 0
fi

TMPD=$(mktemp -d)
printf 'A small known-good seed. LocalSDR sells a local AI sales copilot to Spanish SMBs for 29 euros per month. Agency owners care about GDPR and cost.\n' > "$TMPD/seed.txt"

probe () {
  local label="$1"; shift
  local code
  code=$(curl -sS -o "$TMPD/resp.json" -w '%{http_code}' "$@")
  echo "== $label -> HTTP $code"
  sed -e 's/\\\\n/\n/g' "$TMPD/resp.json" | head -c 600
  echo; echo
}

URL=http://127.0.0.1:5001/api/graph/ontology/generate

probe 'A: files + simulation_requirement' \
  -F "files=@$TMPD/seed.txt" \
  -F "simulation_requirement=Predict how Spanish SMBs react to a local AI sales copilot in the first 60 days." \
  "$URL"

probe 'B: files + requirement' \
  -F "files=@$TMPD/seed.txt" \
  -F "requirement=Predict how Spanish SMBs react to a local AI sales copilot in the first 60 days." \
  "$URL"

probe 'C: file (singular) + requirement' \
  -F "file=@$TMPD/seed.txt" \
  -F "requirement=Predict how Spanish SMBs react to a local AI sales copilot in the first 60 days." \
  "$URL"

probe 'D: JSON body instead of multipart' \
  -H 'Content-Type: application/json' \
  -d '{"text":"LocalSDR sells a local AI sales copilot to Spanish SMBs for 29 euros per month. Owners care about GDPR and cost.","simulation_requirement":"Predict how Spanish SMBs react to a local AI sales copilot in the first 60 days."}' \
  "$URL"

echo '=== handler validation source ==='
f=$(grep -rln "ontology/generate\|generate_ontology\|ontology_generate" "$ROOT/backend/app" --include='*.py' | head -1)
echo "HANDLER FILE: $f"
grep -n -B8 -A60 "def .*ontolog\|/ontology/generate" "$f" | head -120

echo
echo '=== backend log lines near the 400s ==='
grep -nE 'WARNING|ERROR|400|本体|ontology' /content/local-mirofish/logs/mirofish-backend.log | tail -20

In [ ]:
import pathlib, re

TARGET = pathlib.Path(
    "/content/MiroFish-local/backend/app/services/zep_graphiti_impl.py"
)
if not TARGET.exists():
    raise SystemExit(f"Missing file: {TARGET}")

src = TARGET.read_text()
patched = src

patched = re.sub(r"max_tokens\s*=\s*16384", "max_tokens=1024", patched)
patched = re.sub(r"max_completion_tokens\s*=\s*16384", "max_completion_tokens=1024", patched)
patched = patched.replace("'max_tokens': 16384", "'max_tokens': 1024")
patched = patched.replace('"max_tokens": 16384', '"max_tokens": 1024')

if patched != src:
    backup = TARGET.with_suffix(".py.before_token_cap_fix.bak")
    if not backup.exists():
        backup.write_text(src)
    compile(patched, str(TARGET), "exec")
    TARGET.write_text(patched)
    print("✓ Fork source: capped max_tokens 16384 -> 1024")
else:
    print("No literal 16384 token cap in fork source - checking graphiti-core defaults.")

# Graphiti's OpenAI generic client also defaults output to the full context window.
import graphiti_core
GROOT = pathlib.Path(graphiti_core.__file__).parent
hits = []
for p in GROOT.rglob("*.py"):
    try:
        t = p.read_text()
    except Exception:
        continue
    new_t = re.sub(r"max_tokens\s*=\s*16384", "max_tokens=1024", t)
    new_t = new_t.replace("'max_tokens': 16384", "'max_tokens': 1024")
    new_t = new_t.replace('"max_tokens": 16384', '"max_tokens": 1024')
    if new_t != t:
        bak = p.with_suffix(".py.tokcap.bak")
        if not bak.exists():
            bak.write_text(t)
        compile(new_t, str(p), "exec")
        p.write_text(new_t)
        hits.append(str(p))

if hits:
    print("✓ Patched graphiti-core token cap in:")
    for h in hits:
        print("  -", h)
else:
    print("No graphiti-core 16384 max_tokens literals found either.")
    print("If the context-length 400 still appears, grep for the value:")
    print("  grep -rn '16384' /content/MiroFish-local/backend/.venv/lib/python3.12/site-packages/graphiti_core | head")

print("Restart the backend (step 9) to load the change.")


## 9. Start the MiroFish backend (loopback only)

**Do not use `npm run backend` here.** That script executes `uv run python run.py`, and `uv run` re-syncs the project venv to the locked dependency set — silently removing the camel-oasis and mcp<2 packages installed manually in step 7 (the backend would crash minutes later with `No module named 'oasis'`). This cell starts `backend/.venv/bin/python run.py` directly, with `VIRTUAL_ENV` and `PATH` set to exactly what `uv run` would provide.

The backend stays on `127.0.0.1:5001` here; its public tunnel is created at the very end (step 13), after the pipeline is proven locally. Liveness is tracked via the backgrounded PID (`kill -0`), not a cmdline pgrep — the latter can miss during process startup and report a healthy server as dead. The cell aborts with a log tail if the backend process really exits early.

In [ ]:
%%bash
set -euo pipefail
BACKEND=/content/MiroFish-local/backend
VENV="$BACKEND/.venv"
test -x "$VENV/bin/python" || { echo 'backend venv missing - run step 7 first'; exit 1; }
cd "$BACKEND"
pkill -f 'python run\.py' 2>/dev/null || true
sleep 2
export VIRTUAL_ENV="$VENV"
export PATH="$VENV/bin:$PATH"
nohup "$VENV/bin/python" run.py > /content/local-mirofish/logs/mirofish-backend.log 2>&1 &
BACK_PID=$!
echo "[$(date +%H:%M:%S)] Backend starting (pid $BACK_PID: $VENV/bin/python run.py)..."
up=''
for i in $(seq 1 120); do
  if curl -sS --max-time 3 http://127.0.0.1:5001/health >/dev/null 2>&1; then up=1; break; fi
  if ! kill -0 "$BACK_PID" 2>/dev/null; then
    echo 'Backend process exited early - log tail:'
    tail -n 120 /content/local-mirofish/logs/mirofish-backend.log
    exit 1
  fi
  if [ $((i % 15)) -eq 0 ]; then
    echo "[$(date +%H:%M:%S)] waiting for backend ($((i*2))s): $(tail -n 1 /content/local-mirofish/logs/mirofish-backend.log | cut -c1-200)"
  fi
  sleep 2
done
if [ -z "$up" ]; then
  echo 'Backend did not start - log tail:'
  tail -n 120 /content/local-mirofish/logs/mirofish-backend.log
  exit 1
fi
echo "[$(date +%H:%M:%S)] Backend healthy on http://127.0.0.1:5001 (loopback only; tunnel comes in step 13)"
tail -n 10 /content/local-mirofish/logs/mirofish-backend.log

## 10. Start the frontend (loopback only)

The dev server stays on `127.0.0.1:3000`; its public tunnel is created at the very end (step 13).

**Why vite is invoked directly:** the fork's `dev` script is a bare `vite`, and on this Node/Vite 7 stack a bare `vite` binds `localhost` as IPv6 `::1` only — the log line "Network: use --host to expose" while `curl http://127.0.0.1:3000/` gets refused is exactly that. The cell runs `frontend/node_modules/.bin/vite --host 127.0.0.1 --port 3000 --strictPort` itself: deterministic IPv4 bind, no npm wrapper, and a loud failure if the port is occupied.

No API base URL is baked here — the backend has no public URL yet. The frontend uses the local default until step 13 writes the real tunnel URL into `frontend/.env.local` and restarts vite.

In [ ]:
%%bash
set -euo pipefail
cd /content/MiroFish-local
pkill -f vite 2>/dev/null || true
sleep 2
cd frontend
nohup ./node_modules/.bin/vite --host 127.0.0.1 --port 3000 --strictPort > /content/local-mirofish/logs/mirofish-frontend.log 2>&1 &
FE_PID=$!
echo "[$(date +%H:%M:%S)] Frontend starting (pid $FE_PID: vite --host 127.0.0.1 --port 3000 --strictPort)..."
up=''
for i in $(seq 1 60); do
  if curl -fsS --max-time 3 http://127.0.0.1:3000/ >/dev/null 2>&1; then up=1; break; fi
  if ! kill -0 "$FE_PID" 2>/dev/null; then
    echo 'Frontend process exited early - log tail:'
    tail -n 80 /content/local-mirofish/logs/mirofish-frontend.log
    exit 1
  fi
  if [ $((i % 10)) -eq 0 ]; then
    echo "[$(date +%H:%M:%S)] waiting for frontend ($((i*2))s): $(tail -n 1 /content/local-mirofish/logs/mirofish-frontend.log | cut -c1-200)"
  fi
  sleep 2
done
if [ -z "$up" ]; then
  echo 'Frontend did not start - log tail:'
  tail -n 80 /content/local-mirofish/logs/mirofish-frontend.log
  exit 1
fi
echo "[$(date +%H:%M:%S)] Frontend is up on http://127.0.0.1:3000 (loopback only; tunnel comes in step 13)"

## 11a. Debug the ontology 400 (on-demand)

Run this if the UI or demo.py returns 400 on `/api/graph/ontology/generate`. It replays a minimal multipart request against the local backend, prints the **full response body** (which names the rejected field), and dumps the handler's validation block. Requires steps 7-9 to have run in this VM.

In [ ]:
%%bash
set +e
ROOT=/content/MiroFish-local
if [ ! -d "$ROOT/backend/app" ]; then
  echo 'repo not present in THIS VM - run the Session state check first'
  exit 0
fi
if ! curl -sS --max-time 3 http://127.0.0.1:5001/health >/dev/null 2>&1; then
  echo 'backend not answering on 5001 - re-run step 9 first'
  exit 0
fi

echo '=== replay: minimal multipart request, full response body ==='
TMPD=$(mktemp -d)
printf 'A small known-good seed paragraph. LocalSDR sells a local AI sales copilot to Spanish SMBs for 29 euros per month.
' > "$TMPD/seed.txt"
curl -sS -o "$TMPD/resp.json" -w 'HTTP %{http_code}\n' \
  -F "files=@$TMPD/seed.txt" \
  -F "simulation_requirement=Predict how Spanish SMBs react to a local AI sales copilot in the first 60 days." \
  -F "project_name=debug-ontology-400" \
  http://127.0.0.1:5001/api/graph/ontology/generate
echo '--- response body ---'
cat "$TMPD/resp.json"; echo

echo
echo '=== handler validation code ==='
f=$(grep -rln "ontology/generate\|generate_ontology\|ontology_generate" "$ROOT/backend/app" --include='*.py' | head -1)
echo "HANDLER FILE: $f"
grep -n -B6 -A55 "def .*ontolog" "$f" | head -100

echo
echo '=== backend log lines near the 400 ==='
grep -nE 'WARNING|ERROR|400|本体|ontology' /content/local-mirofish/logs/mirofish-backend.log | tail -20

## 12. Prove the graph built

The UI's task log only tells you the task failed, not whether anything landed. The ground truth is Neo4j itself. After a build completes (or while it runs), these cells count nodes and edges and list what was extracted.

In [ ]:
import pathlib

TARGET = pathlib.Path(
    "/content/MiroFish-local/backend/app/services/zep_graphiti_impl.py"
)

src = TARGET.read_text()
backup = TARGET.with_suffix(".py.before_schema_reader_fix.bak")

if not backup.exists():
    backup.write_text(src)

patched = src

# Nodes: accept legacy EntityNode plus common Graphiti labels.
patched = patched.replace(
    "MATCH (n:EntityNode {group_id: $group_id})",
    """MATCH (n)
                    WHERE n.group_id = $group_id
                      AND (
                          n:EntityNode
                          OR n:Entity
                          OR n:Episodic
                      )"""
)

# Edges: accept legacy EntityNode plus modern Graphiti labels.
patched = patched.replace(
    "MATCH (n:EntityNode {group_id: $group_id})-[r]-(m:EntityNode)",
    """MATCH (n)-[r]-(m)
                    WHERE n.group_id = $group_id
                      AND m.group_id = $group_id
                      AND (
                          n:EntityNode
                          OR n:Entity
                          OR n:Episodic
                      )
                      AND (
                          m:EntityNode
                          OR m:Entity
                          OR m:Episodic
                      )"""
)

if patched == src:
    print("No exact legacy EntityNode queries found.")
    print("Print the relevant source first:")
    print("grep -n -B8 -A25 'EntityNode' /content/MiroFish-local/backend/app/services/zep_graphiti_impl.py")
else:
    compile(patched, str(TARGET), "exec")
    TARGET.write_text(patched)
    print("✓ Patched Graphiti adapter reader to accept EntityNode, Entity, and Episodic labels.")
    print("✓ Backup saved:", backup)
    print("Restart the backend next.")

In [ ]:
%%bash
set +e
if docker exec mirofish-neo4j true >/dev/null 2>&1; then
  CYPHER='docker exec mirofish-neo4j cypher-shell'
else
  CYPHER=/content/neo4j/bin/cypher-shell
fi
echo '--- node/edge totals ---'
$CYPHER -u neo4j -p password 'MATCH (n) RETURN count(n) AS nodes' 2>&1
$CYPHER -u neo4j -p password 'MATCH ()-[r]->() RETURN count(r) AS edges' 2>&1
echo '--- per-label counts ---'
$CYPHER -u neo4j -p password 'MATCH (n) RETURN labels(n)[0] AS label, count(*) AS count ORDER BY count DESC LIMIT 15' 2>&1
echo '--- sample entities ---'
$CYPHER -u neo4j -p password 'MATCH (n:Entity) RETURN n.name AS name, n.summary AS summary LIMIT 10' 2>&1

### Reset the graph between experiments

The next cell deletes all nodes and edges. Use it between experiments; vectors and index dims stay consistent because the same bge-m3 model is always used — only change embedding models on an empty database.

In [ ]:
%%bash
set +e
if docker exec mirofish-neo4j true >/dev/null 2>&1; then
  CYPHER='docker exec mirofish-neo4j cypher-shell'
else
  CYPHER=/content/neo4j/bin/cypher-shell
fi
$CYPHER -u neo4j -p password 'MATCH (n) DETACH DELETE n' 2>&1
echo 'Graph cleared.' 

## Appendix: known-good proof seed

This seed (English, short declarative sentences, explicit named entities) is tuned for the 350M extractor. The next cell writes it into the VM so it can be attached from Colab's file browser or used headlessly.

In [ ]:
PROOF_SEED = """LocalSDR: a local-first AI sales copilot for Spanish SMBs

LocalSDR is a small startup based in Sitges, Spain. It sells a sales-automation
copilot that runs on the customer's own hardware: a local language model drafts
and answers WhatsApp sales messages, so no customer data leaves the company.

Product and price. The product costs 29 euros per month per seat. It connects
to WhatsApp Business, qualifies inbound leads, and writes follow-up messages in
Spanish and English. Because inference is local, there is no per-message API
cost and no cloud dependency.

Customers. The first target customers are small real-estate agencies in
Barcelona and Sitges. These agencies answer most leads over WhatsApp and lose
leads when agents reply slowly. Agency owners care about cost and about GDPR
compliance. Their sales agents care about not being replaced and about message
quality.

Competitors. Cloud tools like HubSpot and Salesforce are more powerful but cost
much more. New AI SDR services such as Artisan are cloud-based and charge per
message. LocalSDR is cheaper and private, but its local model is less capable
than large cloud models at writing nuanced messages.

Key risk events. Next month, WhatsApp may raise Business API prices. The EU AI
Act will add compliance duties for AI-generated messages. LocalSDR plans to
launch publicly on Product Hunt in two weeks.

Open question. Will Spanish SMB owners trust and pay for a local AI copilot,
and will sales agents adopt it or resist it?
"""
path = '/content/local-mirofish/proof-seed.txt'
import os
os.makedirs('/content/local-mirofish', exist_ok=True)
with open(path, 'w') as f:
    f.write(PROOF_SEED)
print('Seed written to', path, f'({len(PROOF_SEED)} chars)')
print()
print('Suggested requirement for the UI:')
print('Predict how Spanish SMB owners, sales agents, and competitors react to')
print('LocalSDR in the first 60 days after launch - willingness to pay EUR 29/month,')
print('trust in local on-device AI vs cloud tools, and WhatsApp-channel fatigue.')

## 13. Go public — tunnels last, after the pipeline works

Everything up to here ran on loopback. Only now, with the backend and frontend proven, do we expose them. This cell:

1. Creates the backend quick tunnel, writes its URL to `backend_url.txt`.
2. Bakes that URL into `frontend/.env.local` and restarts vite (Vite reads env at dev-server start — an already-running vite would keep the old default otherwise).
3. Creates the frontend quick tunnel with `--http-host-header localhost:3000` so Vite's host check accepts the trycloudflare domain.

Why tunnels come last: earlier revisions tunnelled the backend at start time, so any backend restart (e.g. after the 8b patch) silently invalidated the frontend's API URL. With tunnels at the end, a backend restart only needs a re-run of this cell.

In [ ]:
%%bash
set -euo pipefail
cd /content/MiroFish-local
for svc in 5001 3000; do
  curl -sS --max-time 3 "http://127.0.0.1:$svc/" >/dev/null 2>&1 \
    || curl -sS --max-time 3 "http://127.0.0.1:$svc/health" >/dev/null 2>&1 \
    || { echo "service on :$svc is not up - run steps 9 and 10 first"; exit 1; }
done

if ! command -v cloudflared >/dev/null 2>&1; then
  echo "[$(date +%H:%M:%S)] Installing cloudflared..."
  curl -fsSL -o /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
  dpkg -i /tmp/cloudflared.deb >/dev/null
fi

pkill -f 'cloudflared tunnel --url http://127.0.0.1:5001' 2>/dev/null || true
sleep 2
nohup cloudflared tunnel --url http://127.0.0.1:5001 > /content/local-mirofish/logs/tunnel-backend.log 2>&1 &
BACKEND_URL=''
for i in $(seq 1 30); do
  BACKEND_URL=$(grep -Eo 'https://[-a-z0-9]+\.trycloudflare\.com' /content/local-mirofish/logs/tunnel-backend.log | head -1 || true)
  [ -n "$BACKEND_URL" ] && break
  sleep 2
done
[ -n "$BACKEND_URL" ] || { tail -n 40 /content/local-mirofish/logs/tunnel-backend.log; exit 1; }
echo "$BACKEND_URL" > /content/local-mirofish/backend_url.txt
echo "Backend tunnel: $BACKEND_URL"

printf 'VITE_API_BASE_URL=%s\n' "$BACKEND_URL" > frontend/.env.local
pkill -f vite 2>/dev/null || true
sleep 2
cd frontend
VITE_API_BASE_URL="$BACKEND_URL" nohup ./node_modules/.bin/vite --host 127.0.0.1 --port 3000 --strictPort > /content/local-mirofish/logs/mirofish-frontend.log 2>&1 &
FE_PID=$!
up=''
for i in $(seq 1 60); do
  if curl -fsS --max-time 3 http://127.0.0.1:3000/ >/dev/null 2>&1; then up=1; break; fi
  kill -0 "$FE_PID" 2>/dev/null || { echo 'vite restart failed:'; tail -n 40 /content/local-mirofish/logs/mirofish-frontend.log; exit 1; }
  sleep 2
done
[ -n "$up" ] || { tail -n 40 /content/local-mirofish/logs/mirofish-frontend.log; exit 1; }
cd /content/MiroFish-local

pkill -f 'cloudflared tunnel --url http://127.0.0.1:3000' 2>/dev/null || true
sleep 2
nohup cloudflared tunnel --url http://127.0.0.1:3000 --http-host-header localhost:3000 > /content/local-mirofish/logs/tunnel-frontend.log 2>&1 &
FRONTEND_URL=''
for i in $(seq 1 30); do
  FRONTEND_URL=$(grep -Eo 'https://[-a-z0-9]+\.trycloudflare\.com' /content/local-mirofish/logs/tunnel-frontend.log | head -1 || true)
  [ -n "$FRONTEND_URL" ] && break
  sleep 2
done
[ -n "$FRONTEND_URL" ] || { tail -n 40 /content/local-mirofish/logs/tunnel-frontend.log; exit 1; }

echo
echo "Open MiroFish: $FRONTEND_URL"
echo "API tunnel:     $BACKEND_URL"
echo 'LLM and embedding APIs remain private on the Colab VM. Unprotected temporary links - do not share.' 

## Diagnostics

Run this cell if graph building or simulation fails. Never publish full logs if they include seed content.

- **Graph build fails with `'NoneType' object has no attribute 'call_soon_threadsafe'`** right after a successful ontology generation: the fork's `_run_async` submits to a never-started event loop. Run step 8b, restart the backend (step 9), rebuild.
- **Ontology 400 within ~1s and no vLLM request at that timestamp**: request-level rejection, not a model failure. Attach an actual TXT/MD seed and fill the requirement field (the endpoint is multipart/form-data). If it persists, run step 11a — it prints the response body and the handler's validation code.
- Router says `{"llm":"down: ..."}` or a proxied call returns 502 `upstream vLLM on :8000 unreachable`: vLLM is dead or was never started in this VM. Check `pgrep -af vllm.entrypoints`, `tail -n 30 .../logs/vllm.log`, and `free -g`. The router needs no restart — it connects per request.
- `No such file or directory` for `/content/MiroFish-local`, `/content/modelenv` or `/content/neo4j`: you are on a fresh VM (reconnect/reset/new upload wipes `/content`). Run the Session state check cell and resume from the earliest missing step.
- Vite prints "ready" but `curl http://127.0.0.1:3000/` never answers: a bare `vite` bound to IPv6 `::1` only. Steps 10/13 start vite with `--host 127.0.0.1 --strictPort` explicitly.
- "process exited early" with an EMPTY log right after a start cell: on revisions before the `$!`-based liveness fix this was a pgrep-vs-startup race, not a real death — check the port before believing it.
- `rm: cannot remove '/content/MiroFish-local': Directory not empty` during step 7: a still-running process is writing into the tree and `rm -rf` loses the race. This revision's step 7 kills tree-anchored processes and moves the tree aside atomically. Manual fix: `mv /content/MiroFish-local /content/MiroFish-local.stale` and re-run.
- A `RuntimeError` about PyTorch/TorchAudio CUDA versions, or `ImportError: libcudart.so.13`, means model code ran outside `/content/modelenv` or the venv flavor drifted — re-run step 3 (it rebuilds the venv when the flavor marker mismatches).
- `Error in sitecustomize: ModuleNotFoundError: No module named 'wrapt'` is a mis-packaged vLLM dependency hook; harmless, silenced by `uv pip install --python /content/modelenv/bin/python wrapt` (already included in step 3 of this revision).
- `failed to create NAT chain DOCKER`, `can't initialize iptables table 'nat'`, `failed to mount ... fstype: bind`, or `failed to register layer: unshare`: Colab's gVisor sandbox blocks containers entirely. Step 6 falls back to a containerless Neo4j automatically — `cat /content/local-mirofish/docker_mode.txt` should say `hostpkg`.
- `ModuleNotFoundError: No module named 'oasis'` (or `graphiti_core`) when the BACKEND starts, after step 7 passed: something re-ran `uv sync`/`uv run`, pruning the manual installs. This revision starts the backend with the venv interpreter directly; never use `npm run backend`/`uv run` after step 7. Re-run step 7 (it now reuses healthy installs) to restore.
- `ImportError: cannot import name 'FastMCP' from 'mcp.server'` means mcp 2.x was installed; camel-ai 0.2.78 needs mcp 1.x. Fix: `uv pip install --python /content/MiroFish-local/backend/.venv/bin/python "mcp>=1.28,<2"`.
- A Neo4j write error mentioning `vector.dimensions` means an embedding-dimension mismatch: confirm the embed server health payload says `"dim": 1024`.
- A vLLM 400/500 often indicates the 350M model could not honor a complex schema. Try a smaller seed and simulation, or move up to `LFM2.5-1.2B-Instruct`.
- A 401/gated error in the vLLM log means the Hugging Face repo needs license acceptance — see step 2.
- A browser request to `localhost:5001` means `VITE_API_BASE_URL` was not picked up: check `frontend/.env.local` and restart the vite process (Vite reads env at dev-server start).
- "Blocked request. This host is not allowed" in the browser means the frontend tunnel lost its `--http-host-header` flag.
- `curl: (22) ... 500` during step 6 is a transient GitHub error downloading the optional APOC jar; the cell retries and Graphiti works without APOC.
- `curl: (23) Failed writing body` under `set -o pipefail` means a `curl | head` pipe closed early — it is cosmetic, not a service failure; this revision removes those pipes.

In [ ]:
%%bash
set +e
echo '--- GPU/driver ---'; nvidia-smi
echo '--- model venv ---'
echo "flavor: $(cat /content/modelenv/.flavor 2>/dev/null || echo 'not installed')"
/content/modelenv/bin/python -c "import torch, vllm; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| vllm', vllm.__version__, '| cuda available:', torch.cuda.is_available())" 2>&1 | tail -3
echo "neo4j mode: $(cat /content/local-mirofish/docker_mode.txt 2>/dev/null || echo 'n/a')"
echo "loop patch: $(grep -c 'global _async_loop' /content/MiroFish-local/backend/app/services/zep_graphiti_impl.py 2>/dev/null || echo 0) marker(s) in zep_graphiti_impl.py"
echo '--- backend venv deps ---'
/content/MiroFish-local/backend/.venv/bin/python -c "import neo4j, graphiti_core, mcp; print('neo4j', neo4j.__version__, '| graphiti-core OK | mcp', getattr(mcp, '__version__', '1.x'))" 2>&1 | tail -3
/content/MiroFish-local/backend/.venv/bin/python -c "import oasis; print('oasis OK')" 2>&1 | tail -3
echo '--- local LLM ---'; tail -n 80 /content/local-mirofish/logs/vllm.log
echo '--- embeddings ---'; tail -n 60 /content/local-mirofish/logs/embed.log
echo '--- router ---'; tail -n 40 /content/local-mirofish/logs/router.log
echo '--- MiroFish backend ---'; tail -n 160 /content/local-mirofish/logs/mirofish-backend.log
echo '--- frontend ---'; tail -n 60 /content/local-mirofish/logs/mirofish-frontend.log
echo '--- Neo4j (docker) ---'; docker logs --tail 60 mirofish-neo4j 2>&1
echo '--- Neo4j (host install) ---'; tail -n 60 /content/local-mirofish/logs/neo4j.log 2>/dev/null || echo 'no host-install log'
echo '--- memory ---'; free -g
echo '--- process/ports ---'; ss -ltnp | grep -E ':(3000|5001|7474|7687|8000|8001|9000)' || true

In [ ]:
%%bash
set -e

/content/neo4j/bin/cypher-shell \
  -u neo4j \
  -p password \
  'MATCH (n) RETURN count(n) AS nodes'

/content/neo4j/bin/cypher-shell \
  -u neo4j \
  -p password \
  'MATCH ()-[r]->() RETURN count(r) AS edges'

/content/neo4j/bin/cypher-shell \
  -u neo4j \
  -p password \
  'MATCH (n) RETURN labels(n)[0] AS label, count(*) AS count ORDER BY count DESC LIMIT 15'